# Einen KI-Agenten mit Tool-Superkräften bauen 🦸 — mit smolagents
_Ursprünglich verfasst von: [Aymeric Roucher](https://huggingface.co/m-ric) — aktualisiert im April 2026_

Dieses Notebook zeigt, wie man mit [**smolagents**](https://huggingface.co/docs/smolagents/index) leistungsstarke **Agenten** bauen kann — wahlweise **lokal mit Ollama** oder **in der Cloud mit OpenRouter**.

### Was sind Agenten?
Agenten sind Systeme, die von einem LLM angetrieben werden. Durch geschicktes Prompting und Ausgabeverarbeitung kann das LLM bestimmte *Tools* nutzen, um Aufgaben zu lösen.

### Was ist smolagents?
Eine Bibliothek von Hugging Face, die Bausteine bereitstellt, um eigene Agenten zu entwickeln! Mehr dazu in der [Dokumentation](https://huggingface.co/docs/smolagents/index).

### Zwei Backends zur Auswahl

| | **Ollama** (lokal) | **OpenRouter** (Cloud) |
|---|---|---|
| API-Key nötig? | Nein | Ja (kostenlos für einige Modelle) |
| Datenschutz | Alles bleibt lokal | Daten gehen an Drittanbieter |
| Geschwindigkeit | Abhängig von GPU/CPU | Schnell (Server-GPUs) |
| Kosten | Keine | Free-Tier oder Pay-per-Use |

Zuerst installieren wir die benötigten Abhängigkeiten:

In [ ]:
import subprocess, sys, importlib.util, time
from IPython.display import clear_output

PACKAGES = [
    ("smolagents[openai]",      "smolagents",             "LLM Agent Framework"),
    ("datasets",                "datasets",               "HuggingFace Datasets"),
    ("langchain",               "langchain",              "LangChain Core"),
    ("langchain-community",     "langchain_community",    "LangChain Community"),
    ("langchain-core",          "langchain_core",         "LangChain Base"),
    ("langchain-text-splitters","langchain_text_splitters","LangChain Text Split"),
    ("langchain-ollama",        "langchain_ollama",       "LangChain Ollama"),
    ("faiss-cpu",               "faiss",                  "Vektordatenbank"),
    ("ddgs",                    "ddgs",                   "Web-Suche (ddgs)"),
    ("openai",                  "openai",                 "OpenAI Client"),
    ("python-dotenv",           "dotenv",                 ".env Unterstützung"),
]

def render(statuses):
    total = len(statuses)
    done  = sum(1 for s in statuses.values() if s["icon"] in ("✅", "❌"))
    bar_filled = int(30 * done / total)
    bar = "█" * bar_filled + "░" * (30 - bar_filled)
    lines = [
        f"📦 Abhängigkeiten  [{bar}]  {done}/{total}\n",
        f"  {'Paket':<26} {'Status':<18} {'Dauer':>6}",
        f"  {'─'*26} {'─'*18} {'─'*6}",
    ]
    for pkg, info in statuses.items():
        dur = f"{info['elapsed']:.0f}s" if info["elapsed"] else ""
        lines.append(f"  {info['label']:<26} {info['icon']} {info['msg']:<15} {dur:>6}")
    clear_output(wait=True)
    print("\n".join(lines), flush=True)

statuses = {
    pkg: {"label": label, "icon": "⏳", "msg": "wartend...", "elapsed": None}
    for pkg, _, label in PACKAGES
}

render(statuses)

for pkg_install, pkg_import, label in PACKAGES:
    statuses[pkg_install]["icon"] = "⏳"
    statuses[pkg_install]["msg"]  = "prüfe..."
    render(statuses)

    already = importlib.util.find_spec(pkg_import) is not None
    if already:
        statuses[pkg_install]["icon"] = "✅"
        statuses[pkg_install]["msg"]  = "bereits vorhanden"
        render(statuses)
        continue

    statuses[pkg_install]["msg"] = "installiere..."
    render(statuses)

    t0 = time.time()
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", pkg_install],
        capture_output=True, text=True,
    )
    elapsed = time.time() - t0
    statuses[pkg_install]["elapsed"] = elapsed

    if result.returncode == 0:
        statuses[pkg_install]["icon"] = "✅"
        statuses[pkg_install]["msg"]  = "installiert"
    else:
        statuses[pkg_install]["icon"] = "❌"
        statuses[pkg_install]["msg"]  = "FEHLER!"
    render(statuses)

errors = [p for p, s in statuses.items() if s["icon"] == "❌"]
if errors:
    print(f"\n❌ Fehler bei: {', '.join(errors)}")
else:
    print("\n✅ Alle Pakete bereit — Kernel muss ggf. neu gestartet werden.")


## ⚙️ Konfiguration — Backend wählen

Hier wählst du, welches Backend du nutzen möchtest. Entweder direkt in der Zelle oder über eine `.env`-Datei (siehe `.env.example`).

**Ollama:** Stelle sicher, dass [Ollama](https://ollama.com/) läuft und ein Modell geladen ist (`ollama pull gemma4:e4b`).

**OpenRouter:** Trage deinen API-Key ein (kostenlos unter [openrouter.ai](https://openrouter.ai)).

In [ ]:
import os
from dotenv import load_dotenv

print("🔧 Lade Konfiguration...")
load_dotenv()  # Liest .env falls vorhanden

# ┌─────────────────────────────────────────────────────────────────┐
# │  BACKEND-AUSWAHL: "ollama" oder "openrouter"                   │
# │  Kann hier direkt gesetzt oder per .env-Datei konfiguriert     │
# │  werden (Variable LLM_BACKEND).                                │
# └─────────────────────────────────────────────────────────────────┘
LLM_BACKEND = os.getenv("LLM_BACKEND", "ollama")  # ← hier ändern oder in .env setzen

# Ollama-Einstellungen
OLLAMA_MODEL_ID = os.getenv("OLLAMA_MODEL_ID", "gemma4:e4b")
OLLAMA_API_BASE = os.getenv("OLLAMA_API_BASE", "http://localhost:11434/v1")

# OpenRouter-Einstellungen
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY", "")
OPENROUTER_MODEL_ID = os.getenv("OPENROUTER_MODEL_ID", "nvidia/nemotron-3-super-120b-a12b:free")

print(f"✅ Backend: {LLM_BACKEND}")
if LLM_BACKEND == "ollama":
    print(f"   Modell: {OLLAMA_MODEL_ID}")
    print(f"   API:    {OLLAMA_API_BASE}")
else:
    print(f"   Modell: {OPENROUTER_MODEL_ID}")
    print(f"   Key:    {'***' + OPENROUTER_API_KEY[-4:] if len(OPENROUTER_API_KEY) > 4 else '⚠️ NICHT GESETZT'}")

### Modell initialisieren

Beide Backends nutzen `OpenAIServerModel` — Ollama und OpenRouter bieten beide eine OpenAI-kompatible API.

Für **gemma4** empfiehlt Google/Ollama diese Sampling-Parameter:

| Parameter | Empfehlung | Erklärung |
|---|---|---|
| `temperature` | `1.0` | Standardwert lt. Gemma-4-Docs — höhere Kreativität |
| `top_p` | `0.95` | Nucleus Sampling |
| `top_k` | `64` | Ollama-spezifisch, via `extra_body` |

> **Hinweis:** `top_k` ist kein OpenAI-Standard-Parameter. Für Ollama wird er über `extra_body` übergeben und von der `/v1/chat/completions`-Schnittstelle weitergereicht.


In [ ]:
print("🤖 Initialisiere Modell...")

from smolagents import OpenAIServerModel

if LLM_BACKEND == "ollama":
    model = OpenAIServerModel(
        model_id=OLLAMA_MODEL_ID,
        api_base=OLLAMA_API_BASE,
        api_key="ollama",  # Ollama braucht keinen echten Key, aber das Feld muss gesetzt sein
        # Gemma 4 empfohlene Sampling-Parameter (https://ollama.com/library/gemma4)
        temperature=1.0,
        top_p=0.95,
        extra_body={"options": {"top_k": 64}},  # Ollama-spezifisch
    )
elif LLM_BACKEND == "openrouter":
    if not OPENROUTER_API_KEY or OPENROUTER_API_KEY == "your_api_key_here":
        raise ValueError("⚠️ OPENROUTER_API_KEY ist nicht gesetzt! Bitte in .env eintragen.")
    model = OpenAIServerModel(
        model_id=OPENROUTER_MODEL_ID,
        api_base="https://openrouter.ai/api/v1",
        api_key=OPENROUTER_API_KEY,
        temperature=1.0,
        top_p=0.95,
    )
else:
    raise ValueError(f"⚠️ Unbekanntes Backend: '{LLM_BACKEND}'. Nutze 'ollama' oder 'openrouter'.")

# Schnelltest: Funktioniert die Verbindung?
print("🔌 Teste Verbindung...")
response = model([{"role": "user", "content": "Say hello briefly!"}])
print(f"✅ Modell antwortet: {response.content[:100]}")


## 1. 🌐 Web-Such-Assistent

Für diesen Anwendungsfall bauen wir einen Agenten, der das Internet durchsuchen kann.

Wir nutzen dafür das eingebaute `DuckDuckGoSearchTool` — das braucht keinen API-Schlüssel und funktioniert sofort.

> **Hinweis zum Agent-Typ:** Hier verwenden wir `ToolCallingAgent` statt `CodeAgent`. Der `ToolCallingAgent` kommuniziert über **JSON Tool Calls** (OpenAI-Format), was lokale Modelle zuverlässiger beherrschen als das Code-Format (`<code>`-Tags) das `CodeAgent` erwartet.

| Agent-Typ | Ausgabeformat | Wann nutzen? |
|---|---|---|
| `CodeAgent` | Python-Code in `<code>`-Tags | Komplexe Logik, Berechnungen, Schleifen |
| `ToolCallingAgent` | JSON Tool Calls | Einfache Tool-Aufrufe, lokale Modelle |

Der Agent bekommt eine Frage, entscheidet selbstständig ob er suchen muss, führt die Suche durch und fasst die Ergebnisse zusammen.

In [ ]:
print("🔧 Erstelle Web-Such-Agent (ToolCallingAgent)...")
from smolagents import ToolCallingAgent, DuckDuckGoSearchTool

search_tool = DuckDuckGoSearchTool()

# ToolCallingAgent nutzt JSON Tool Calls (OpenAI-Format) —
# robuster für lokale Modelle als der CodeAgent-Ansatz mit <code>-Tags.
agent = ToolCallingAgent(
    tools=[search_tool],
    model=model,
)
print("✅ Agent bereit! Starte Anfrage...")

result = agent.run(
    "Which country holds the EU Council Presidency in April 2026?"
)
print("\n📋 Ergebnis:")
print(result)


### Was ist passiert?

Der Agent hat:
1. Die Frage analysiert und erkannt, dass er aktuelle Informationen braucht.
2. Selbstständig eine DuckDuckGo-Suche formuliert und durchgeführt.
3. Die Ergebnisse ausgewertet und eine Antwort formuliert.

Das ist das **ReAct-Muster** (Reason + Act) in Aktion: Der Agent denkt nach, handelt, beobachtet das Ergebnis und wiederholt das bei Bedarf.

## 2. 📚💬 RAG mit iterativer Abfrage-Verfeinerung

**Retrieval-Augmented Generation (RAG)** bedeutet: _"Ein LLM beantwortet Nutzerfragen, basiert die Antwort aber auf Informationen aus einer Wissensdatenbank."_

Vorteile gegenüber einem normalen LLM:
- Antworten werden auf echte Fakten gestützt → weniger Halluzinationen
- Domänenspezifisches Wissen wird eingebunden
- Feingranulare Kontrolle über den Informationszugang

### Das Problem
Oft liefert eine einfache RAG-Suche keine relevanten Ergebnisse. Was dann?

### Unsere Lösung
Wir geben dem **Agenten die Kontrolle über die Suchparameter!** Er kann:
- Die Suche auf bestimmte Quellen einschränken
- Die Anzahl der abgerufenen Dokumente anpassen
- Bei schlechten Ergebnissen die Suche mit angepasster Query wiederholen

Laden wir zuerst eine Wissensbasis — hier eine Sammlung von Dokumentationsseiten verschiedener Hugging-Face-Pakete:

In [ ]:
print("📥 Lade Wissensbasis von Hugging Face Hub... (kann beim ersten Mal etwas dauern)")
import datasets

knowledge_base = datasets.load_dataset("m-ric/huggingface_doc", split="train")
print(f"✅ Wissensbasis geladen: {len(knowledge_base)} Dokumente")

### Vektordatenbank aufbauen

Jetzt verarbeiten wir die Wissensbasis und speichern sie in einer FAISS-Vektordatenbank. Wir nutzen LangChain für die Text-Aufteilung und Ollama für die Embeddings.

> **Voraussetzung:** Das Embedding-Modell muss in Ollama verfügbar sein — unabhängig davon, welches LLM-Backend du nutzt:
> ```
> ollama pull nomic-embed-text
> ```
> `nomic-embed-text` ist ein schnelles, qualitativ hochwertiges Embedding-Modell (~274MB).

> **Warum immer Ollama für Embeddings?** OpenRouter bietet keine Embeddings-API an — Embeddings laufen in RAG-Systemen typischerweise lokal. Auch bei OpenRouter als LLM-Backend werden Embeddings lokal über Ollama berechnet. Das ist sogar ein Vorteil: keine Kosten, keine Latenz, keine Datenweitergabe.


In [ ]:
print("📄 Importiere LangChain-Komponenten...")
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings

print("📄 Erstelle Dokumente aus Wissensbasis...")
source_docs = [
    Document(page_content=doc["text"], metadata={"source": doc["source"].split("/")[1]})
    for doc in knowledge_base
]

print("✂️ Teile Dokumente in Chunks auf...")
docs_processed = RecursiveCharacterTextSplitter(chunk_size=500).split_documents(
    source_docs
)[:1000]

# Embeddings laufen immer lokal über Ollama — unabhängig vom LLM-Backend (Ollama oder OpenRouter).
# OpenRouter bietet keine Embeddings-API. OLLAMA_API_BASE ist immer gesetzt (Standard: localhost:11434).
print("🧮 Lade Embedding-Modell (nomic-embed-text via lokales Ollama)...")
ollama_base_url = OLLAMA_API_BASE.replace("/v1", "")  # http://localhost:11434
embedding_model = OllamaEmbeddings(
    model="nomic-embed-text",
    base_url=ollama_base_url,
)

print("🗃️ Baue FAISS-Vektordatenbank... (kann 1-2 Minuten dauern)")
vectordb = FAISS.from_documents(documents=docs_processed, embedding=embedding_model)

print(f"✅ Vektordatenbank erstellt mit {len(docs_processed)} Chunks")


### Verfügbare Quellen

Unsere Dokumentationsseiten stammen aus verschiedenen Paketen. Schauen wir uns an, welche Quellen wir haben:

In [ ]:
all_sources = list(set([doc.metadata["source"] for doc in docs_processed]))
print(f"Verfügbare Quellen ({len(all_sources)}):")
for source in sorted(all_sources):
    print(f"  - {source}")

### Eigenes Retriever-Tool bauen

Jetzt bauen wir ein `RetrieverTool`, das der Agent nutzen kann, um Informationen aus der Wissensbasis abzurufen.

Da wir die Vektordatenbank als Attribut brauchen, erstellen wir eine Tool-Klasse (statt den einfachen `@tool`-Decorator zu nutzen). So sieht die fortgeschrittene Tool-Definition aus:

In [ ]:
import json
from smolagents import Tool
from langchain_core.vectorstores import VectorStore


class RetrieverTool(Tool):
    name = "retriever"
    description = "Searches the knowledge base and returns documents whose embeddings are most similar to the query."
    inputs = {
        "query": {
            "type": "string",
            "description": "The search query. Should be semantically close to the target documents. Use affirmative statements rather than questions.",
        },
        "source": {
            "type": "string",
            "description": "",
            "nullable": True,  # smolagents v1.24: required for optional parameters
        },
        "number_of_documents": {
            "type": "string",
            "description": "Number of documents to retrieve. Maximum 10 to avoid overloading the context.",
            "nullable": True,  # smolagents v1.24: required for optional parameters
        },
    }
    output_type = "string"

    def __init__(self, vectordb: VectorStore, all_sources: list[str], **kwargs):
        super().__init__(**kwargs)
        self.vectordb = vectordb
        self.inputs["source"]["description"] = (
            f"The source(s) to search as a string representation of a list. "
            f"Possible values: {all_sources}. If not specified, all sources are searched.".replace(
                "'", "`"
            )
        )

    def forward(self, query: str, source: str = None, number_of_documents=7) -> str:
        assert isinstance(query, str), "Query must be a string"
        number_of_documents = int(number_of_documents) if number_of_documents else 7

        if source:
            if isinstance(source, str) and "[" not in str(source):
                source = [source]
            source = json.loads(str(source).replace("'", '"'))

        docs = self.vectordb.similarity_search(
            query,
            filter=({"source": source} if source else None),
            k=number_of_documents,
        )

        if len(docs) == 0:
            return "No documents found. Try again without a source filter."
        return "Retrieved documents:\n\n" + "\n===Document===\n".join(
            [doc.page_content for doc in docs]
        )


### Tool testen

Bevor wir das Tool an den Agenten übergeben, testen wir es kurz manuell:

In [ ]:
print("🔧 Erstelle RetrieverTool...")
retriever_tool = RetrieverTool(vectordb=vectordb, all_sources=all_sources)

# Manueller Test: Dokumente zum Tokenizer-Thema abrufen
print("🔍 Teste manuelle Suche nach 'AutoTokenizer text tokenization'...")
test_result = retriever_tool.forward("AutoTokenizer text tokenization", number_of_documents=3)
print(f"✅ {len(test_result)} Zeichen abgerufen. Vorschau:")
print(test_result[:500])


### Den RAG-Agenten starten!

Jetzt verbinden wir alles: Wir erstellen einen `ToolCallingAgent` mit unserem Retriever-Tool und dem lokalen Ollama-Modell.

In [ ]:
print("🤖 Erstelle RAG-Agent...")
from smolagents import ToolCallingAgent

retriever_tool = RetrieverTool(vectordb=vectordb, all_sources=all_sources)

rag_agent = ToolCallingAgent(
    tools=[retriever_tool],
    model=model,
)
print("✅ RAG-Agent bereit! Starte Anfrage...")

agent_output = rag_agent.run(
    "How do I load a pretrained model and tokenizer using the Transformers library? "
    "Show me a short example with AutoTokenizer and AutoModel."
)

print("\n📋 Ergebnis:")
print(agent_output)


### Was ist passiert?

Der Agent hat den Retriever mit spezifischen Quellen aufgerufen (z.B. `['transformers', 'blog']`).

Falls die erste Suche nicht genug Ergebnisse liefert, kann der Agent **iterieren** — er passt die Suchparameter an und versucht es erneut. Das ist das Schöne an einem Agenten-gesteuerten RAG: Es ist **flexibler als klassisches RAG**, weil der Agent die Suchstrategie dynamisch anpassen kann.

## 3. 💻 Python-Code debuggen

Da der `CodeAgent` einen eingebauten Python-Interpreter hat, können wir ihn nutzen, um fehlerhaften Code zu debuggen!

In [ ]:
print("🤖 Erstelle Debug-Agent...")
from smolagents import CodeAgent

debug_agent = CodeAgent(
    tools=[],
    model=model,
    code_block_tags="markdown",
    max_steps=6,
    instructions=(
        "Always write your code inside a markdown code block: ```python ... ```. "
        "End every run by calling final_answer() with your result."
    ),
)
print("✅ Agent bereit!")

fehlerhafter_code = """
zahlen = [0, 1, 2]

for i in range(4):
    print(zahlen(i))
"""

print("🐛 Übergebe fehlerhaften Code an den Agenten...")
final_answer = debug_agent.run(
    "I have code that causes an error. Please debug it, "
    "run it to make sure it works, "
    "and return the corrected code.",
    additional_args={"code": fehlerhafter_code},
)
print("\n📋 Ergebnis:")
print(final_answer)


Der Agent hat:
1. Den Code ausgeführt und den Fehler erkannt (`zahlen(i)` statt `zahlen[i]` — runde statt eckige Klammern)
2. Den Code korrigiert
3. Den korrigierten Code getestet
4. Das Ergebnis zurückgegeben

Der korrigierte Code:

In [ ]:
print(final_answer)

## 4. 🔧 Eigene Tools erstellen

Eigene Tools zu bauen ist einfach. Mit dem `@tool`-Decorator reicht eine Funktion mit Docstring — smolagents extrahiert daraus automatisch Name, Beschreibung und Parameter für den Agenten.

In [ ]:
print("🔧 Definiere eigene Tools...")
from smolagents import tool, CodeAgent

@tool
def word_count(text: str) -> str:
    """Counts the words in a text and returns a summary.

    Args:
        text: The text whose words should be counted.
    """
    words = text.split()
    chars = len(text)
    sentences = text.count(".") + text.count("!") + text.count("?")
    return f"The text contains {len(words)} words, {chars} characters and approx. {sentences} sentences."


@tool
def caesar_encrypt(text: str, shift: int) -> str:
    """Encrypts a text using the Caesar cipher.

    Args:
        text: The text to encrypt.
        shift: The number of positions to shift each letter.
    """
    result = []
    for char in text:
        if char.isalpha():
            base = ord("A") if char.isupper() else ord("a")
            result.append(chr((ord(char) - base + shift) % 26 + base))
        else:
            result.append(char)
    return "".join(result)

print("✅ Tools definiert: word_count, caesar_encrypt")

# Agent mit unseren eigenen Tools
print("🤖 Erstelle Agent mit eigenen Tools...")
tool_agent = CodeAgent(
    tools=[word_count, caesar_encrypt],
    model=model,
    code_block_tags="markdown",
    max_steps=6,
    instructions=(
        "Always write your code inside a markdown code block: ```python ... ```. "
        "End every run by calling final_answer() with your result."
    ),
)
print("✅ Agent bereit! Starte Anfrage...")

result = tool_agent.run(
    "Encrypt the sentence 'Hello World' with Caesar shift 3, "
    "then count the words in the encrypted text."
)
print("\n📋 Ergebnis:")
print(result)


## 5. 🤝 Multi-Agenten-System

Mehrere Agenten können zusammenarbeiten, um komplexere Aufgaben zu lösen. Das Konzept stammt aus Microsofts [Autogen-Framework](https://huggingface.co/papers/2308.08155) und liefert auf vielen Benchmarks bessere Ergebnisse.

Die Idee: Statt eines "Alleskönner"-Agenten spezialisieren wir einzelne Agenten auf Teilaufgaben. Ein **Manager-Agent** koordiniert die Arbeit und delegiert an spezialisierte Unter-Agenten.

In [ ]:
print("🤖 Erstelle Multi-Agenten-System...")
from smolagents import ToolCallingAgent, DuckDuckGoSearchTool

# Spezialisierter Web-Such-Agent (ToolCallingAgent — gleicher Grund wie in Abschnitt 1)
web_agent = ToolCallingAgent(
    tools=[DuckDuckGoSearchTool()],
    model=model,
    name="web_search_agent",
    description="Performs web searches. Pass your search query as the argument.",
    max_steps=3,
)
print("  ✅ Web-Such-Agent erstellt")

# Manager-Agent, der den Web-Agenten koordiniert.
# Fix: gemma4 tends to call managed agents with "arguments=..." instead of "task=...".
# The managed agent tool schema always uses "task" as the parameter name.
manager_agent = ToolCallingAgent(
    tools=[],
    model=model,
    managed_agents=[web_agent],
    max_steps=5,
    instructions=(
        "When calling a sub-agent tool, always pass the query using the 'task' parameter. "
        "Example: web_search_agent(task='your search query here'). "
        "Never use 'arguments' as a parameter name."
    ),
)
print("  ✅ Manager-Agent erstellt")
print("🚀 Starte Multi-Agenten-Anfrage...")

result = manager_agent.run(
    "Find out which new features were introduced in Python 3.13 "
    "and summarize the three most important ones."
)
print("\n📋 Ergebnis:")
print(result)


## 6. 🎨 Bonus: Gradio-Oberfläche

smolagents bringt eine eingebaute Gradio-UI mit, über die man interaktiv mit dem Agenten chatten kann:

In [ ]:
# Auskommentiert — bei Bedarf einkommentieren und ausführen.
# Öffnet eine Web-Oberfläche im Browser.

# from smolagents import CodeAgent, DuckDuckGoSearchTool, GradioUI
#
# chat_agent = CodeAgent(
#     tools=[DuckDuckGoSearchTool()],
#     model=model,
# )
#
# GradioUI(chat_agent).launch()

## ➡️ Fazit

Dieses Notebook hat gezeigt, wie man mit **smolagents v1.24** leistungsstarke KI-Agenten bauen kann — wahlweise lokal mit **Ollama** oder in der Cloud mit **OpenRouter**:

| Beispiel | Was wir gelernt haben |
|---|---|
| **Konfiguration** | Dual-Backend: Ollama (lokal) oder OpenRouter (Cloud) per Variable umschaltbar |
| **Web-Suche** | Agent + DuckDuckGo für aktuelle Informationen |
| **RAG** | Agenten-gesteuertes Retrieval mit iterativer Verfeinerung |
| **Code-Debugging** | CodeAgent mit eingebautem Python-Interpreter |
| **Eigene Tools** | `@tool`-Decorator für schnelle Tool-Erstellung |
| **Multi-Agenten** | Spezialisierte Agenten unter einem Manager |
| **Gradio-UI** | Interaktive Chat-Oberfläche |

### Weiterführende Ressourcen
- [smolagents Dokumentation](https://huggingface.co/docs/smolagents/index)
- [Ollama Modell-Bibliothek](https://ollama.com/library)
- [OpenRouter Modelle](https://openrouter.ai/models)
- [Tools-Tutorial](https://huggingface.co/docs/smolagents/tutorials/tools)
- [Gute Agenten bauen](https://huggingface.co/docs/smolagents/tutorials/building_good_agents)
- [Sichere Code-Ausführung](https://huggingface.co/docs/smolagents/tutorials/secure_code_execution)

Viel Spaß beim Experimentieren! 🚀